# 🌍 GIEC Chat - AR6 (RAG avec citations)

**Projet IA / Climat - Rapport GIEC AR6 WG1**

Notebook unique regroupant tout le pipeline :
1. **Ingestion** PDF -> chunks
2. **Vectorisation** chunks -> FAISS (bge-m3)
3. **Requetage** RAG + App Streamlit

> Donnees : https://www.ipcc.ch/report/ar6/wg1/ -> Placer `IPCC_AR6_WGI_Chapter01.pdf` dans `data/raw/`


## 0. Mettre Google drive comme répertoire de travail par défaut

In [4]:
#from google.colab import drive
#drive.mount('/content/drive')

#%cd "/content/drive/MyDrive/Colab Notebooks/Climate Change - GIEC chat"
%cd "/home/dominique/GIEC-Chat"

/home/dominique/GIEC-Chat


## 1. Installation
A exécuter une fois

In [5]:
# Installation des dépendances
# Installe depuis requirements.txt (recommandé) avec fallback
import pathlib, subprocess
UV = "/home/dominique/.local/bin/uv"
REPO = pathlib.Path("/home/dominique/GIEC-Chat")
req = REPO / "requirements.txt"
if req.exists():
    print(f"📦 Installation depuis {req} ...")
    #subprocess.check_call([UV, "pip", "install", "-r", str(req)])
else:
    print("⚠️ requirements.txt non trouvé -> fallback install")
    subprocess.check_call([UV, "pip", "install", "pypdf", "sentence-transformers", "faiss-cpu", "streamlit", "requests"])
print("Dépendances installées")
try:
    import pypdf, sentence_transformers, faiss
    print("✅ Vérif imports OK")
except Exception as e:
    print(f"⚠️ Vérif échouée: {e} -> relance le runtime si besoin")


📦 Installation depuis /home/dominique/GIEC-Chat/requirements.txt ...
Dépendances installées
✅ Vérif imports OK


## 2. Conversion des PDF -> chunks
Découpe les chapitres en chunks de 800 tokens avec overlap 100. Conserve la page source pour citation.


In [6]:
import json
import re
import requests
from pathlib import Path
from tqdm.auto import tqdm
try:
    from pypdf import PdfReader, overwrite_configuration, apply_configuration
except ModuleNotFoundError:
    import subprocess
    print("[INFO] pypdf manquant -> uv pip install ...")
    subprocess.check_call(["/home/dominique/.local/bin/uv", "pip", "install", "-q", "pypdf"])
    from pypdf import PdfReader, overwrite_configuration, apply_configuration
    print("✅ pypdf installé")

from pypdf.errors import LimitReachedError

# Augmente le budget d'analyse des content streams (PDFs AR6 très verbeux).
overwrite_configuration(stream_parsing_work_maximum_count=100_000_000)

# --- Config multi-chapitres (1-12 + SPM) ---
BASE_URL = "https://www.ipcc.ch/report/ar6/wg1/downloads/report/"
CHAPTERS = {
    1: "IPCC_AR6_WGI_Chapter01.pdf",
    2: "IPCC_AR6_WGI_Chapter02.pdf",
    3: "IPCC_AR6_WGI_Chapter03.pdf",
    4: "IPCC_AR6_WGI_Chapter04.pdf",
    5: "IPCC_AR6_WGI_Chapter05.pdf",
    6: "IPCC_AR6_WGI_Chapter06.pdf",
    7: "IPCC_AR6_WGI_Chapter07.pdf",
    8: "IPCC_AR6_WGI_Chapter08.pdf",
    9: "IPCC_AR6_WGI_Chapter09.pdf",
    10: "IPCC_AR6_WGI_Chapter10.pdf",
    11: "IPCC_AR6_WGI_Chapter11.pdf",
    12: "IPCC_AR6_WGI_Chapter12.pdf",
    "SPM": "IPCC_AR6_WGI_SPM.pdf",
}
CHAPTER_TITLES = {
    1: "Ch.1 - Smoking gun", 2: "Ch.2 - Changing state", 3: "Ch.3 - Human influence",
    4: "Ch.4 - Future climate", 5: "Ch.5 - Carbon cycle", 6: "Ch.6 - Extremes",
    7: "Ch.7 - Clouds & aerosols", 8: "Ch.8 - Water cycle", 9: "Ch.9 - Ocean & cryosphere",
    10: "Ch.10 - Regional", 11: "Ch.11 - Extremes regional", 12: "Ch.12 - Climate information for risk",
    "SPM": "Summary for Policymakers"
}

def extract_with_metadata(pdf_path: Path, chapter_id):
    """Extrait chaque page avec pypdf, enrichit metadata avec chapter."""
    reader = PdfReader(str(pdf_path))
    chunks = []
    title = CHAPTER_TITLES.get(chapter_id, f"Ch.{chapter_id}")
    source_label = f"AR6 WG1 Ch.{chapter_id}" if isinstance(chapter_id, int) else f"AR6 WG1 {chapter_id}"
    for i, page in enumerate(reader.pages):
        try:
            text = page.extract_text() or ""
        except LimitReachedError:
            print(f"  [WARN] page {i + 1}: budget par defaut depasse, retry avec budget x10")
            with apply_configuration(stream_parsing_work_maximum_count=1_000_000_000):
                text = page.extract_text() or ""
        text = re.sub(r'\n{2,}', '\n', text).strip()
        if len(text) < 300:
            continue
        chunks.append({
            "text": text,
            "metadata": {
                "source": source_label,
                "chapter": chapter_id,
                "chapter_title": title,
                "page": i + 1,
            }
        })
    return chunks

def decouper(texte, taille=800, chevauchement=100):
    """Découpe sans langchain : ~800 chars, overlap 100, coupe en fin de phrase."""
    texte = " ".join(texte.split())
    passages, debut = [], 0
    while debut < len(texte):
        fin = min(debut + taille, len(texte))
        if fin < len(texte):
            coupe = texte.rfind(". ", debut + taille // 2, fin)
            if coupe != -1:
                fin = coupe + 1
        passages.append(texte[debut:fin].strip())
        if fin >= len(texte):
            break
        debut = max(fin - chevauchement, debut + 1)
    return [p for p in passages if p]

def split_chunks(chunks, chunk_size=800, chunk_overlap=100):
    final = []
    for c in chunks:
        for chunk in decouper(c["text"], taille=chunk_size, chevauchement=chunk_overlap):
            final.append({"text": chunk, "metadata": c["metadata"]})
    return final

# --- Boucle full 1-12 + SPM ---
ALL_CHUNKS = []
for ch_id, fname in tqdm(CHAPTERS.items(), desc="Chapitres", unit="pdf"):
    pdf_path = Path(f"data/raw/{fname}")
    # naming conservé : data/processed/chunks_chapterX.json
    out_name = f"chunks_chapter{ch_id}.json" if isinstance(ch_id, int) else f"chunks_{str(ch_id).lower()}.json"
    out_path = Path(f"data/processed/{out_name}")
    print(f"\n--- Chapitre {ch_id} : {fname} ---")
    if not pdf_path.exists():
        url = BASE_URL + fname
        print(f"[INFO] PDF non trouvé: {pdf_path} -> téléchargement {url} ...")
        pdf_path.parent.mkdir(parents=True, exist_ok=True)
        try:
            r = requests.get(url, timeout=120)
            r.raise_for_status()
            pdf_path.write_bytes(r.content)
            print(f"✅ Téléchargé : {pdf_path} ({len(r.content)/1e6:.1f} MB)")
        except Exception as e:
            print(f"[WARN] Échec téléchargement Ch.{ch_id} : {e} -> skip")
            continue
    else:
        print(f"✅ PDF présent : {pdf_path}")
    print(f"Extraction de {pdf_path} ...")
    raw = extract_with_metadata(pdf_path, ch_id)
    print(f"  {len(raw)} pages extraites")
    final = split_chunks(raw)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(final, f, ensure_ascii=False, indent=2)
    print(f"  {len(final)} chunks -> {out_path}")
    ALL_CHUNKS.extend(final)

# Agrégat unifié pour l'index unique (non committé)
ALL_PATH = Path("data/processed/chunks_all.json")
with open(ALL_PATH, "w", encoding="utf-8") as f:
    json.dump(ALL_CHUNKS, f, ensure_ascii=False, indent=2)
print(f"\n✅ Total {len(ALL_CHUNKS)} chunks (tous chapitres) -> {ALL_PATH}")
# Stats par chapitre
from collections import Counter
cnt = Counter(c["metadata"]["chapter"] for c in ALL_CHUNKS)
for k in sorted(cnt, key=lambda x: (isinstance(x,str), x)):
    print(f"  Ch.{k}: {cnt[k]} chunks")


Chapitres:   0%|          | 0/13 [00:00<?, ?pdf/s]


--- Chapitre 1 : IPCC_AR6_WGI_Chapter01.pdf ---
✅ PDF présent : data/raw/IPCC_AR6_WGI_Chapter01.pdf
Extraction de data/raw/IPCC_AR6_WGI_Chapter01.pdf ...


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/IDDLHD+MyriadPro-Regular', '/Encoding': IndirectObject(18041, 0, 125052484441040), '/FirstChar': 28, '/FontDescriptor': IndirectObject(18043, 0, 125052484441040), '/LastChar': 31, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [332, 305, 347, 311]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/CPBBBP+MyriadPro-Cond', '/Encoding': IndirectObject(20281, 0, 125052484441040), '/FirstChar': 30, '/FontDescriptor': IndirectObject(20283, 0, 125052484441040), '/LastChar': 31, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [270, 245]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
Chapitres:   8%|▊         | 1/13 [00:12<02:30, 12.57s/pdf]

  140 pages extraites
  1341 chunks -> data/processed/chunks_chapter1.json

--- Chapitre 2 : IPCC_AR6_WGI_Chapter02.pdf ---
✅ PDF présent : data/raw/IPCC_AR6_WGI_Chapter02.pdf
Extraction de data/raw/IPCC_AR6_WGI_Chapter02.pdf ...


Chapitres:  15%|█▌        | 2/13 [00:31<02:58, 16.24s/pdf]

  136 pages extraites
  1322 chunks -> data/processed/chunks_chapter2.json

--- Chapitre 3 : IPCC_AR6_WGI_Chapter03.pdf ---
✅ PDF présent : data/raw/IPCC_AR6_WGI_Chapter03.pdf
Extraction de data/raw/IPCC_AR6_WGI_Chapter03.pdf ...


... (18 outputs left out)

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/VCYUPN+MyriadPro-Regular', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 51, '/FontDescriptor': IndirectObject(514, 0, 125052357626192), '/LastChar': 120, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [513, 0, 513, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 463]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
Chapitres:  92%|█████████▏| 12/13 [05:31<00:25, 25.05s/pdf]

  159 pages extraites
  1650 chunks -> data/processed/chunks_chapter12.json

--- Chapitre SPM : IPCC_AR6_WGI_SPM.pdf ---
✅ PDF présent : data/raw/IPCC_AR6_WGI_SPM.pdf
Extraction de data/raw/IPCC_AR6_WGI_SPM.pdf ...


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/BTEIVM+MyriadPro-Regular', '/Encoding': IndirectObject(294, 0, 125052402000496), '/FirstChar': 31, '/FontDescriptor': IndirectObject(295, 0, 125052402000496), '/LastChar': 146, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [523, 212, 0, 0, 0, 0, 0, 0, 0, 284, 284, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 493, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 482, 569, 448, 564, 501, 292, 559, 555, 234, 0, 0, 236, 834, 555, 549, 0, 0, 327, 396, 331, 551, 481, 736, 0, 471, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 207]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
Chapitres: 100%|██████████| 13/13 [05:35<00:00, 25.85s/pdf]

  29 pages extraites
  212 chunks -> data/processed/chunks_spm.json

✅ Total 17740 chunks (tous chapitres) -> data/processed/chunks_all.json
  Ch.1: 1341 chunks
  Ch.2: 1322 chunks
  Ch.3: 1219 chunks
  Ch.4: 1142 chunks
  Ch.5: 1398 chunks
  Ch.6: 1015 chunks
  Ch.7: 1358 chunks
  Ch.8: 1573 chunks
  Ch.9: 1537 chunks
  Ch.10: 1571 chunks
  Ch.11: 2402 chunks
  Ch.12: 1650 chunks
  Ch.SPM: 212 chunks


# 3. Conversion des chunks -> embeddings

In [ ]:
import json
import faiss
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

# Source unifiée (tous chapitres) — per-chapter conservés en data/processed/chunks_chapter*.json
CHUNKS_PATH = Path("data/processed/chunks_all.json")
# Fallback si chunks_all pas encore généré (ex: run Ch.1 seul)
if not CHUNKS_PATH.exists():
    fallback = Path("data/processed/chunks_chapter1.json")
    print(f"[INFO] {CHUNKS_PATH} absent -> fallback {fallback}")
    CHUNKS_PATH = fallback
VECTORSTORE_PATH = Path("vectorstore/giec_wg1_all")
EMBEDDING_MODEL = "BAAI/bge-m3"

if not CHUNKS_PATH.exists():
    raise FileNotFoundError(f"[ERREUR] {CHUNKS_PATH} introuvable. Lance d'abord la cellule 4 (ingestion full).")

with open(CHUNKS_PATH, encoding="utf-8") as f:
    data_json = json.load(f)

texts = [d["text"] for d in data_json]
metadatas = [d["metadata"] for d in data_json]

print(f"Vectorisation de {len(texts)} chunks (tous chapitres) avec {EMBEDDING_MODEL} ...")
model = SentenceTransformer(EMBEDDING_MODEL)

embeddings = model.encode(texts, normalize_embeddings=True, show_progress_bar=True, batch_size=128)
# On normalise chaque embedding pour que sa norme soit égale à 1. Dans ce cas, la similarité cosinus entre deux vecteurs A et B est : A produit scalaire B
embeddings = np.array(embeddings, dtype="float32")

index = faiss.IndexFlatIP(embeddings.shape[1]) # embeddings.shape[1] = 1024 pour BGE-m3 ; crée un index vide configuré pour recevoir des vecteurs de 1024 dimensions et les comparer avec le produit scalaire.
# IndexFlatIP = Produit scalaire
# IndexFlatL2 = Distance euclidienne au carré
# J'utilise IndexFlatIP car les embeddings sont normalisés, donc la distance cosinus est équivalente au produit scalaire.
index.add(embeddings) # ajout des embeddings à l'index FAISS
VECTORSTORE_PATH.mkdir(parents=True, exist_ok=True)
faiss.write_index(index, str(VECTORSTORE_PATH / "index.faiss"))
with open(VECTORSTORE_PATH / "chunks.json", "w", encoding="utf-8") as f:
    json.dump(data_json, f, ensure_ascii=False, indent=2)
# Stats par chapitre pour vérif
from collections import Counter
cnt = Counter(m["chapter"] for m in metadatas)
print(f"Index FAISS unifié -> {VECTORSTORE_PATH} ({index.ntotal} vecteurs, {embeddings.shape[1]} dims)")
for k in sorted(cnt, key=lambda x: (isinstance(x,str), x)):
    print(f"  Ch.{k}: {cnt[k]} chunks")

# Test cross-chapitres
for query in ["Quelle est la part du réchauffement d'origine humaine ?", "Probabilité de dépasser 1.5°C avant 2040 ?"]:
    q_emb = model.encode([query], normalize_embeddings=True)
    scores, ids = index.search(np.array(q_emb, dtype="float32"), k=3)
    # La proximité n'est pas une propriété mathématique présente dans les phrases elles-mêmes : c'est une propriété de la représentation que le modèle a apprise pendant son entraînement.
    print(f"\nTest: {query}")
    for rank, (idx, score) in enumerate(zip(ids[0], scores[0]), 1):
        print(f"  {rank}. [score={score:.3f}] Ch.{metadatas[idx]['chapter']} p.{metadatas[idx]['page']} {metadatas[idx]['source']}")
        print(f"     {texts[idx][:180]}...")


print("Modèle :", EMBEDDING_MODEL)
print("Dimensions du modèle :", model.get_embedding_dimension())
print("Shape des embeddings :", embeddings.shape)
print("Nombre de vecteurs FAISS :", index.ntotal)
print("Type des données :", embeddings.dtype)
print("Mémoire utilisée (MiB) :", embeddings.nbytes / (1024**2))
print("GPU ou CPU ?", model.device)


Vectorisation de 17740 chunks (tous chapitres) avec BAAI/bge-m3 ...


Batches: 100%|██████████| 139/139 [4:08:43<00:00, 107.36s/it] 


Index FAISS unifié -> vectorstore/giec_wg1_all (17740 vecteurs, 1024 dims)
  Ch.1: 1341 chunks
  Ch.2: 1322 chunks
  Ch.3: 1219 chunks
  Ch.4: 1142 chunks
  Ch.5: 1398 chunks
  Ch.6: 1015 chunks
  Ch.7: 1358 chunks
  Ch.8: 1573 chunks
  Ch.9: 1537 chunks
  Ch.10: 1571 chunks
  Ch.11: 2402 chunks
  Ch.12: 1650 chunks
  Ch.SPM: 212 chunks

Test: Quelle est la part du réchauffement d'origine humaine ?
  1. [score=0.677] Ch.1 p.140 AR6 WG1 Ch.1
     e anthropogenic increase in greenhouse gas concentrations and other anthropogenic forcings together. The best estimate of the human-induced contribution to warming is similar to th...
  2. [score=0.672] Ch.3 p.94 AR6 WG1 Ch.3
     516 Chapter 3 Human Influence on the Climate System 3 FAQ 3.1 (continued) FAQ 3.1: How do we know humans are causing climate change? Observed warming (1850-2019) is only reproduced...
  3. [score=0.670] Ch.1 p.38 AR6 WG1 Ch.1
     t of the warming observed over the last 50 years is attributable to human activities’ (I

In [ ]:
import os

print(os.path.exists("data/processed/chunks_chapter1.json"))
print(os.path.abspath("data/processed/chunks_chapter1.json"))

True
/content/drive/MyDrive/Colab Notebooks/Climate Change - GIEC chat/data/processed/chunks_chapter1.json


# Code complet app.py - Streamlit unifié 1-12+SPM (sans langchain)
import json
import faiss
import numpy as np
import streamlit as st
from pathlib import Path
from sentence_transformers import SentenceTransformer

VECTORSTORE_PATH = Path("vectorstore/giec_wg1_all")
if not (VECTORSTORE_PATH / "index.faiss").exists():
    VECTORSTORE_PATH = Path("vectorstore/giec_wg1_ch1")
EMBEDDING_MODEL = "BAAI/bge-m3"

SYSTEM_PROMPT = """Tu es l'assistant GIEC. Réponds UNIQUEMENT avec le contexte fourni.
Cite obligatoirement [Source: WG1 Ch.X/SPM, p.Y]. Si la réponse n'est pas dans le contexte, dis "Non trouvé dans le rapport". N'invente rien."""

st.set_page_config(page_title="GIEC Chat - AR6 WG1 (1-12+SPM)", layout="wide")
st.title("🌍 GIEC Chat - AR6 WG1 (Ch.1-12 + SPM)")
st.caption("RAG unifié avec citations vérifiables - Données: IPCC AR6 WG1")

@st.cache_resource
def load_db():
    if not (VECTORSTORE_PATH / "index.faiss").exists():
        return None, None, None, None
    index = faiss.read_index(str(VECTORSTORE_PATH / "index.faiss"))
    with open(VECTORSTORE_PATH / "chunks.json", encoding="utf-8") as f:
        data = json.load(f)
    texts = [d["text"] for d in data]
    metadatas = [d["metadata"] for d in data]
    model = SentenceTransformer(EMBEDDING_MODEL)
    return index, texts, metadatas, model

index, texts, metadatas, model = load_db()

if index is None:
    st.warning("Index unifié non trouvé. Lance les cellules 4 et 6 (full 1-12+SPM) après téléchargement des PDFs dans `data/raw/`.")
    st.info("PDFs auto-téléchargés depuis: https://www.ipcc.ch/report/ar6/wg1/downloads/report/")
    st.stop()

# Filtre chapitre
chapters = sorted(set(m["chapter"] for m in metadatas), key=lambda x: (isinstance(x,str), x))
sel = st.selectbox("Filtrer par chapitre", ["Tous"] + [f"Ch.{c}" for c in chapters])
query = st.text_input("Pose ta question au rapport du GIEC :", placeholder="Ex: Quelle est la probabilité de dépasser 1.5°C avant 2040 ?")

if query:
    q_emb = model.encode([query], normalize_embeddings=True)
    # k=5 pour couvrir plusieurs chapitres
    scores, ids = index.search(np.array(q_emb, dtype="float32"), k=10)
    # Filtrage si sélection
    docs = []
    for rank, idx in enumerate(ids[0]):
        meta = metadatas[idx]
        if sel != "Tous" and f"Ch.{meta['chapter']}" != sel:
            continue
        docs.append((texts[idx], meta, float(scores[0][rank])))
        if len(docs) >= 5:
            break
    col1, col2 = st.columns([2, 1])
    with col1:
        st.subheader("Réponse (à brancher à un LLM)")
        st.info(f"{SYSTEM_PROMPT}\n\nContexte récupéré pour: '{query}' (filtre: {sel})")
        for i, (text, meta, score) in enumerate(docs, 1):
            st.markdown(f"**Source {i} - Ch.{meta['chapter']} p.{meta['page']} [score={score:.3f}]**")
            st.write(text)
            st.divider()
    with col2:
        st.subheader("Sources vérifiables")
        for _, meta, _ in docs:
            st.markdown(f"- **{meta['source']}**, p.{meta['page']} ({meta['chapter_title']})")
        st.caption("Index unifié giec_wg1_all - Branche un LLM (Mistral) pour génération avec citations.")


In [ ]:
import json
import faiss
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

VECTORSTORE_PATH = Path("vectorstore/giec_wg1_all")
EMBEDDING_MODEL = "BAAI/bge-m3"

if not (VECTORSTORE_PATH / "index.faiss").exists():
    print("Index non trouvé. Lance d'abord les cellules d'ingestion (4) et d'embeddings (8).")
else:
    index = faiss.read_index(str(VECTORSTORE_PATH / "index.faiss"))
    with open(VECTORSTORE_PATH / "chunks.json", encoding="utf-8") as f:
        data = json.load(f)
    texts = [d["text"] for d in data]
    metadatas = [d["metadata"] for d in data]
    model = SentenceTransformer(EMBEDDING_MODEL)
    question = "Quelle est la part du réchauffement d'origine humaine ?"
    question_embeddings = model.encode([question], normalize_embeddings=True)
    scores, ids = index.search(np.array(question_embeddings, dtype="float32"), k=3)
    for rank, (idx, score) in enumerate(zip(ids[0], scores[0]), 1):
        print(f"--- Resultat {rank} [score={score:.3f}] {metadatas[idx]} ---")
        print(texts[idx][:600])
        print()


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

--- Resultat 1 [score=0.677] {'source': 'AR6 WG1 Chapitre 1', 'page': 140} ---
e anthropogenic increase in greenhouse gas concentrations and other anthropogenic forcings together. The best estimate of the human-induced contribution to warming is similar to the observed warming over this period. It is unequivocal that human influence has warmed the atmosphere, ocean and land. The likely range of total human-caused global surface temperature increase from 1850–1900 to 2010–201911 is 0.8°C to 1.3°C, with a best estimate of 1.07°C. It is likely that well-mixed GHGs contributed a warming of 1.0°C to 2.0°C, other human drivers (principally aerosols) contributed a cooling of 0.

--- Resultat 2 [score=0.670] {'source': 'AR6 WG1 Chapitre 1', 'page': 38} ---
t of the warming observed over the last 50 years is attributable to human activities’ (IPCC, 2001b). The AR4 further strengthened previous statements, concluding that ‘most of the observed increase in global average temperatures since the mi

## 4. LLM Groq Cloud — génération avec citations (stream)
Appel `groq/compound` via Groq, clé dans `.env` (`GROQ_API_KEY`), fallback `qwen/qwen3.8-27b`, `stream=True`. Si la réponse n'est pas dans le contexte → `Non trouvé dans le rapport`.


In [13]:
# 4. LLM Groq Cloud - groq/compound (stream=True, .env)
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()  # charge .env à la racine du repo si présent

# Clé via .env ou Colab userdata ou saisie interactive (jamais hardcodée)
if not os.getenv("GROQ_API_KEY"):
    try:
        from google.colab import userdata
        _k = userdata.get("GROQ_API_KEY")
        if _k: os.environ["GROQ_API_KEY"] = _k
    except Exception:
        pass
if not os.getenv("GROQ_API_KEY"):
    import getpass
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY (https://console.groq.com/keys) : ").strip()

from groq import Groq
import faiss, json, numpy as np
from sentence_transformers import SentenceTransformer

GROQ_MODEL_PRIMARY = "groq/compound"
GROQ_MODEL_FALLBACK = "qwen/qwen3.8-27b"
VECTORSTORE_PATH = Path("vectorstore/giec_wg1_all")
if not (VECTORSTORE_PATH / "index.faiss").exists():
    VECTORSTORE_PATH = Path("vectorstore/giec_wg1_ch1")
EMBEDDING_MODEL = "BAAI/bge-m3"

print(f"Chargement index {VECTORSTORE_PATH} ...")
index = faiss.read_index(str(VECTORSTORE_PATH / "index.faiss"))
with open(VECTORSTORE_PATH / "chunks.json", encoding="utf-8") as f:
    _data = json.load(f)
texts = [d["text"] for d in _data]
metadatas = [d["metadata"] for d in _data]
embedder = SentenceTransformer(EMBEDDING_MODEL)
client = Groq()
print(f"✅ Index {index.ntotal} vecteurs + Groq client prêt ({GROQ_MODEL_PRIMARY})")

SYSTEM_PROMPT_FR = """Tu es l'assistant GIEC. Réponds UNIQUEMENT avec le contexte fourni. Cite obligatoirement [Source: WG1 Ch.X/SPM, p.Y]. Si la réponse n'est pas dans le contexte, dis exactement \"Non trouvé dans le rapport\". N'invente rien. Réponds en français."""
SYSTEM_PROMPT_EN = """You are the IPCC assistant. Answer ONLY from the provided context. Cite [Source: WG1 Ch.X/SPM, p.Y]. If not in context, say exactly \"Not found in the report\". Do not hallucinate. Answer in English."""

def _detect_lang(q: str) -> str:
    # heuristique simple FR/EN
    fr_markers = ["quelle","quel","pourquoi","comment","est-ce","réchauffement","chapitre"]
    return "fr" if any(w in q.lower() for w in fr_markers) else "en"

def rechercher(query: str, k: int = 5):
    q_emb = embedder.encode([query], normalize_embeddings=True)
    scores, ids = index.search(np.array(q_emb, dtype="float32"), k=k)
    docs = [(texts[i], metadatas[i], float(scores[0][rank])) for rank, i in enumerate(ids[0])]
    return docs

def repondre_avec_llm(query: str, k: int = 5, temperature: float = 0, stream: bool = True):
    docs = rechercher(query, k=k)
    contexte = "\n\n".join(f"[Source: {m['source']}, p.{m['page']}]\n{t[:1200]}" for t, m, _ in docs)
    lang = _detect_lang(query)
    system_prompt = SYSTEM_PROMPT_FR if lang == "fr" else SYSTEM_PROMPT_EN
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Contexte:\n{contexte}\n\nQuestion: {query}"}
    ]
    # Appel Groq avec fallback + gestion 413 payload too large
    def _call(model, msgs):
        return client.chat.completions.create(model=model, messages=msgs, temperature=temperature, max_tokens=600, stream=stream)
    def _build_messages(docs_subset, max_chars=1000):
        ctx = "\n\n".join(f"[Source: {m['source']}, p.{m['page']}]\n{t[:max_chars]}" for t, m, _ in docs_subset)
        return [{"role": "system", "content": system_prompt}, {"role": "user", "content": f"Contexte:\n{ctx}\n\nQuestion: {query}"}]
    # limite contexte total ~4000 chars pour éviter 413
    if len(contexte) > 4500:
        # tronque en gardant k=3 et 800 chars
        docs = docs[:3]
        contexte = "\n\n".join(f"[Source: {m['source']}, p.{m['page']}]\n{t[:800]}" for t, m, _ in docs)
        messages = _build_messages(docs, 800)
    try:
        resp = _call(GROQ_MODEL_PRIMARY, messages)
    except Exception as e:
        msg = str(e).lower()
        if "413" in msg or "request_too_large" in msg or "too large" in msg:
            print(f"⚠️ 413 payload -> retry k=3/800 chars: {e}")
            docs_retry = docs[:3]
            msgs_retry = _build_messages(docs_retry, 800)
            try:
                resp = _call(GROQ_MODEL_PRIMARY, msgs_retry)
            except Exception as e2:
                print(f"⚠️ retry failed -> fallback {GROQ_MODEL_FALLBACK}: {e2}")
                resp = _call(GROQ_MODEL_FALLBACK, msgs_retry)
        elif "rate" in msg or "429" in msg or "quota" in msg:
            print(f"⚠️ {GROQ_MODEL_PRIMARY} quota -> fallback {GROQ_MODEL_FALLBACK}: {e}")
            resp = _call(GROQ_MODEL_FALLBACK, messages)
        else:
            raise
    if not stream:
        return resp.choices[0].message.content, docs
    # mode stream: générateur
    def _gen():
        for chunk in resp:
            delta = chunk.choices[0].delta.content
            if delta: yield delta
    return _gen(), docs

# --- Tests ---
from tqdm.auto import tqdm
for q in tqdm(["Quelle est la part du réchauffement d'origine humaine ?", "Probability to exceed 1.5°C before 2040?", "Quel est le cours du Bitcoin ?"]):
    print(f"\n❓ {q}")
    docs = rechercher(q, k=3)
    for t,m,s in docs: print(f"  - Ch.{m['chapter']} p.{m['page']} score={s:.3f}")
    try:
        gen, _ = repondre_avec_llm(q, k=3, stream=False)
        # si stream=False, gen est str; si stream=True, c'est un générateur -> on force non-stream pour test
        if isinstance(gen, str):
            print(f"  💬 {gen[:400]}...")
        else:
            print(f"  💬 (stream) {' '.join(list(gen))[:400]}...")
    except Exception as e:
        print(f"  ⚠️ LLM non testé (clé manquante ou quota): {e}")


Chargement index vectorstore/giec_wg1_all ...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ Index 17741 vecteurs + Groq client prêt (groq/compound)

❓ Quelle est la part du réchauffement d'origine humaine ?
  - Ch.1 p.140 score=0.677
  - Ch.3 p.94 score=0.672
  - Ch.1 p.38 score=0.670
  💬 **Part du réchauffement d’origine humaine**

- Le réchauffement total attribuable aux activités humaines entre 1850‑1900 et 2010‑2019 est estimé entre **0,8 °C et 1,3 °C**, avec une meilleure estimation de **1,07 °C**.  
  [Source: AR6 WG1 Ch.1, p.140]

- Cette contribution humaine se décompose ainsi :  
  - **Gaz à effet de serre bien mélangés** : apportent un réchauffement de **1,0 °C à 2,0 °C**...

❓ Probability to exceed 1.5°C before 2040?
  - Ch.4 p.3 score=0.670
  - Ch.4 p.31 score=0.667
  - Ch.SPM p.15 score=0.665
  💬 **Probability of exceeding the 1.5 °C global‑warming threshold before 2040**

| Scenario (SSP) | Likelihood that the 1.5 °C level is exceeded sometime in the near‑term window 2021‑2040 |
|----------------|----------------------------------------------------------------

## 5. App Streamlit (optionnel - pour deploiement local)
Code complet pour `streamlit run app.py`. Copie integree ci-dessous pour reference.


In [ ]:
# Code complet app.py - reference / deploiement Streamlit (sans langchain)
import json
import faiss
import numpy as np
import streamlit as st
from pathlib import Path
from sentence_transformers import SentenceTransformer

VECTORSTORE_PATH = Path("vectorstore/giec_wg1_ch1")
EMBEDDING_MODEL = "BAAI/bge-m3"

SYSTEM_PROMPT = """Tu es l'assistant GIEC. Réponds UNIQUEMENT avec le contexte fourni.
Cite obligatoirement [Source: WG1 Ch1, p.X]. Si la réponse n'est pas dans le contexte, dis "Non trouvé dans le rapport". N'invente rien."""

st.set_page_config(page_title="GIEC Chat - AR6 WG1", layout="wide")
st.title("🌍 GIEC Chat - AR6 WG1 Chapitre 1")
st.caption("RAG avec citations vérifiables - Données: IPCC AR6 WG1 (sans langchain)")

@st.cache_resource
def load_db():
    if not (VECTORSTORE_PATH / "index.faiss").exists():
        return None, None, None, None
    index = faiss.read_index(str(VECTORSTORE_PATH / "index.faiss"))
    with open(VECTORSTORE_PATH / "chunks.json", encoding="utf-8") as f:
        data = json.load(f)
    texts = [d["text"] for d in data]
    metadatas = [d["metadata"] for d in data]
    model = SentenceTransformer(EMBEDDING_MODEL)
    return index, texts, metadatas, model

index, texts, metadatas, model = load_db()

if index is None:
    st.warning("Index non trouvé. Lance les cellules 4 et 6 après avoir placé/téléchargé le PDF dans `data/raw/`.")
    st.info("Source PDF auto-téléchargée depuis: https://www.ipcc.ch/report/ar6/wg1/downloads/report/IPCC_AR6_WGI_Chapter01.pdf")
    st.stop()

question = st.text_input("Pose ta question au rapport du GIEC :", placeholder="Ex: Quelle est la probabilité de dépasser 1.5°C avant 2040 ?")

if question:
    question_embeddings = model.encode([question], normalize_embeddings=True)
    scores, ids = index.search(np.array(question_embeddings, dtype="float32"), k=3)
    docs = [(texts[i], metadatas[i], float(scores[0][rank])) for rank, i in enumerate(ids[0])]

    col1, col2 = st.columns([2, 1])

    with col1:
        st.subheader("Réponse (à brancher à un LLM)")
        st.info(f"{SYSTEM_PROMPT}\n\nContexte récupéré pour: '{question}'")
        for i, (text, meta, score) in enumerate(docs, 1):
            st.markdown(f"**Source {i} - {meta} [score={score:.3f}]**")
            st.write(text)
            st.divider()

    with col2:
        st.subheader("Sources vérifiables")
        for _, meta, _ in docs:
            st.markdown(f"- **{meta['source']}**, p.{meta['page']}")
        st.caption("Branche ensuite un LLM (Mistral/Llama) pour générer la réponse finale avec citations obligatoires.")

# Voir app.py (Groq streaming + .env) pour version déployable


---
### Roadmap
- [ ] Placer `IPCC_AR6_WGI_Chapter01.pdf` dans `data/raw/`
- [ ] Executer sections 1 et 2
- [ ] Tester section 3
- [ ] Brancher LLM avec prompt citation obligatoire
